In [ ]:
'''
MFCC + SVM speaker recognition
Deep learning model (CNN/LSTM)
ECAPA-TDNN / speaker embeddings

MFCCs capture the unique vocal characteristics of a speaker.
13 coefficients are standard because they represent the most significant spectral information without being too noisy.
For each frame (short window of audio, e.g., 20–40 ms), you get a 13-dimensional vector.
'''

# ML

In [9]:
import os
import shutil
import librosa
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from IPython.display import Audio, display

import os
import librosa
import soundfile as sf

In [ ]:
# import os
# import librosa
# import soundfile as sf

# # Dataset path
# dataset_path = "/home/shanin/Desktop/SHANIN/MAIN/ALL_CODE/Speaker-Identification/Dataset"

# # Output folder
# output_dir = "/home/shanin/Desktop/SHANIN/MAIN/ALL_CODE/Speaker-Identification/Dataset/Combined_Audio"
# os.makedirs(output_dir, exist_ok=True)

# # Class folders
# speaker_folders = ["Aiyub", "Himel", "Shanin"]

# for speaker in speaker_folders:
#     folder_path = os.path.join(dataset_path, speaker)
    
#     # Get all MP3 files
#     audio_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".mp3")])

#     combined_audio = []
#     sr = None
    
#     for file in audio_files:
#         file_path = os.path.join(folder_path, file)
#         audio, sr = librosa.load(file_path, sr=None)  # Load mp3
#         combined_audio.extend(audio)
    
#     # Save as WAV (better for ML)
#     output_path = os.path.join(output_dir, f"{speaker}_combined.wav")
#     sf.write(output_path, combined_audio, sr)
    
#     print(f"✅ Combined for: {speaker} -> {output_path}")

# print("🎉 Done — all classes combined!")


In [ ]:
# # -------------------------------
# # Combine audio files per class (optional)
# # -------------------------------
# for class_name in class_folders:
#     class_path = os.path.join(dataset_path, class_name)
#     combined_audio = []
#     sr = None
    
#     for filename in sorted(os.listdir(class_path)):
#         if filename.endswith((".wav", ".mp3")):
#             file_path = os.path.join(class_path, filename)
#             audio, sr = librosa.load(file_path, sr=None)
#             combined_audio.extend(audio)
    
#     # Save combined file (optional)
#     combined_path = os.path.join(output_dir, f"{class_name}_combined.wav")
#     sf.write(combined_path, combined_audio, sr)
#     print(f"✅ Combined audio saved: {combined_path}")
# print("🎉 All classes combined!")

In [ ]:
# # -------------------------------
# #  Visualize audio (optional)
# # -------------------------------
# def plot_audio_features(audio_path):
#     y, sr = librosa.load(audio_path, sr=None)
#     class_name = os.path.basename(audio_path).replace("_combined.wav", "")
    
#     plt.figure(figsize=(15,10))
    
#     # Waveform
#     plt.subplot(3,1,1)
#     librosa.display.waveshow(y, sr=sr)
#     plt.title(f"Waveform - {class_name}")
    
#     # Spectrogram
#     plt.subplot(3,1,2)
#     D = librosa.amplitude_to_db(librosa.stft(y), ref=np.max)
#     librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
#     plt.colorbar(format='%+2.0f dB')
#     plt.title(f"Spectrogram - {class_name}")
    
#     # MFCCs
#     plt.subplot(3,1,3)
#     mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
#     librosa.display.specshow(mfccs, x_axis='time', cmap='coolwarm')
#     plt.colorbar()
#     plt.title(f"MFCCs - {class_name}")
    
#     plt.tight_layout()
#     plt.show()

# # Plot for combined audios (optional)
# for class_name in class_folders:
#     combined_path = os.path.join(output_dir, f"{class_name}_combined.wav")
#     plot_audio_features(combined_path)

In [ ]:
# -------------------------------
# Set paths for your dataset
# -------------------------------
dataset_path = "/home/shanin/Downloads/data"


# List of class folders (update names as per your dataset)
class_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
print("Classes found:", class_folders)


Classes found: ['Benjamin_Netanyau', 'Julia_Gillard', 'Magaret_Tarcher', 'Nelson_Mandela', 'Jens_Stoltenberg']


In [26]:
# -------------------------------
#  Extract MFCC features per file
# -------------------------------
def extract_features(dataset_path, class_folders, max_len=100):
    X, y = [], []
    
    for idx, class_name in enumerate(class_folders):
        class_path = os.path.join(dataset_path, class_name)
        for filename in os.listdir(class_path):
            if filename.endswith((".wav", ".mp3")):
                file_path = os.path.join(class_path, filename)
                audio, sr = librosa.load(file_path, sr=None)
                
                mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
                mfccs = StandardScaler().fit_transform(mfccs)
                mfccs = mfccs.T
                
                # Pad/truncate to max_len frames
                if mfccs.shape[0] < max_len:
                    pad_width = max_len - mfccs.shape[0]
                    mfccs = np.pad(mfccs, ((0,pad_width),(0,0)), mode='constant')
                else:
                    mfccs = mfccs[:max_len,:]
                
                X.append(mfccs)
                y.append(idx)
    
    return np.array(X), np.array(y)

X, y = extract_features(dataset_path, class_folders, max_len=100)
print("X shape:", X.shape, "y shape:", y.shape)

X shape: (7451, 100, 13) y shape: (7451,)


In [27]:
# -------------------------------
# Train SVM Classifier
# -------------------------------
# Flatten features for SVM
X_flat = X.reshape(X.shape[0], -1)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

# Train SVM
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train, y_train)

# Predictions
y_pred = svm.predict(X_test)

# Accuracy & Report
print("Test Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=class_folders))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.971830985915493


In [28]:
# Compute confusion matrix
conf_mat = confusion_matrix(y_test, y_pred)

# Display text version
print("Confusion Matrix:\n", conf_mat)


Confusion Matrix:
 [[304   0   6   0   5]
 [  0 302   2   0   1]
 [  1   1 280   0   1]
 [  0   2   0 286   0]
 [ 15   3   5   0 277]]


In [29]:
import librosa
import numpy as np

def predict_speaker(file_path, svm_model, class_folders, max_len=100):
    # Load audio
    audio, sr = librosa.load(file_path, sr=None)
    
    # Extract MFCCs
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfccs = StandardScaler().fit_transform(mfccs)
    mfccs = mfccs.T
    
    # Pad or truncate to max_len
    if mfccs.shape[0] < max_len:
        pad_width = max_len - mfccs.shape[0]
        mfccs = np.pad(mfccs, ((0,pad_width),(0,0)), mode='constant')
    else:
        mfccs = mfccs[:max_len,:]
    
    # Flatten for SVM
    mfccs_flat = mfccs.reshape(1, -1)
    
    # Predict
    pred_label_idx = svm_model.predict(mfccs_flat)[0]
    pred_class = class_folders[pred_label_idx]
    
    return pred_class


In [34]:
# Example test audio file
test_audio_path = "/home/shanin/Downloads/test/nelson/36.wav"

predicted_class = predict_speaker(test_audio_path, svm, class_folders)
print(f"Predicted Class: {predicted_class}")


Predicted Class: Nelson_Mandela


# CNN-LSTM

In [14]:
import os
import librosa
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, BatchNormalization
from tensorflow.keras.utils import to_categorical


2025-11-03 15:52:29.928238: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762163549.994440 3028057 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762163550.012854 3028057 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-03 15:52:30.149592: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [16]:
def extract_mfcc_features(dataset_path, class_folders, n_mfcc=13, max_len=100):
    X, y = [], []
    
    for idx, class_name in enumerate(class_folders):
        class_path = os.path.join(dataset_path, class_name)
        for filename in os.listdir(class_path):
            if filename.endswith((".wav", ".mp3")):
                file_path = os.path.join(class_path, filename)
                audio, sr = librosa.load(file_path, sr=None)
                
                mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
                mfccs = StandardScaler().fit_transform(mfccs)
                mfccs = mfccs.T  # shape: (frames, n_mfcc)
                
                # Pad or truncate
                if mfccs.shape[0] < max_len:
                    pad_width = max_len - mfccs.shape[0]
                    mfccs = np.pad(mfccs, ((0,pad_width),(0,0)), mode='constant')
                else:
                    mfccs = mfccs[:max_len,:]
                
                X.append(mfccs)
                y.append(idx)
    
    return np.array(X), np.array(y)


In [17]:
import os

dataset_path = "/home/shanin/Downloads/data"

# Automatically detect all speaker/class folders
class_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
print(f"Detected {len(class_folders)} classes:", class_folders)

X, y = extract_mfcc_features(dataset_path, class_folders, n_mfcc=13, max_len=100)
print("X shape:", X.shape, "y shape:", y.shape)

# Encode labels
y_encoded = to_categorical(y, num_classes=len(class_folders))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


Detected 5 classes: ['Benjamin_Netanyau', 'Julia_Gillard', 'Magaret_Tarcher', 'Nelson_Mandela', 'Jens_Stoltenberg']
X shape: (7451, 100, 13) y shape: (7451,)


In [ ]:
model = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    MaxPooling1D(2),
    Dropout(0.3),
    Conv1D(128, 3, activation='relu'),
    MaxPooling1D(2),
    Dropout(0.3),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(len(class_folders), activation='softmax')
])


model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/home/shanin/miniconda3/envs/shanin/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 64)         │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 47, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 23, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2944)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       188,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 216,069 (844.02 KB)

 Trainable params: 216,069 (844.02 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

Epoch 1/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9910 - loss: 0.0286 - val_accuracy: 0.9824 - val_loss: 0.0571
Epoch 2/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9952 - loss: 0.0201 - val_accuracy: 0.9715 - val_loss: 0.0829
Epoch 3/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9846 - loss: 0.0392 - val_accuracy: 0.9916 - val_loss: 0.0238
Epoch 4/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9953 - loss: 0.0164 - val_accuracy: 0.9908 - val_loss: 0.0327
Epoch 5/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9957 - loss: 0.0104 - val_accuracy: 0.9832 - val_loss: 0.0520
Epoch 6/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9911 - loss: 0.0220 - val_accuracy: 0.9908 - val_loss: 0.0276
Epoch 7/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9957 - loss: 0.0122 - val_accuracy: 0.9924 - val_loss: 0.0275
Epoch 8/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9948 - loss: 0.0146 - val_accu

In [22]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9960 - loss: 0.0211
Test Accuracy: 0.9912810325622559


In [23]:
import librosa
import numpy as np
from sklearn.preprocessing import StandardScaler

def predict_speaker(audio_path, model, class_folders, n_mfcc=13, max_len=100):
    """
    Predict speaker class from a single audio file.
    """
    # Load audio
    audio, sr = librosa.load(audio_path, sr=None)
    
    # Extract MFCC
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    mfccs = StandardScaler().fit_transform(mfccs)
    mfccs = mfccs.T  # shape: (frames, n_mfcc)
    
    # Pad or truncate to max_len
    if mfccs.shape[0] < max_len:
        pad_width = max_len - mfccs.shape[0]
        mfccs = np.pad(mfccs, ((0,pad_width),(0,0)), mode='constant')
    else:
        mfccs = mfccs[:max_len,:]
    
    # Add batch dimension for model
    mfccs = np.expand_dims(mfccs, axis=0)  # shape: (1, max_len, n_mfcc)
    
    # Predict
    pred_prob = model.predict(mfccs)
    pred_idx = np.argmax(pred_prob, axis=1)[0]
    pred_class = class_folders[pred_idx]
    
    return pred_class, pred_prob[0]


In [ ]:
# Path to new audio file
test_audio_path = "/home/shanin/Downloads/test/maga/12.wav"

# Predict using trained LSTM or CNN model
predicted_class, class_probabilities = predict_speaker(test_audio_path, model, class_folders)

print(f"Predicted Speaker: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
Predicted Speaker: Magaret_Tarcher
Class Probabilities: [1.5421411e-10 7.0837472e-05 9.9992645e-01 1.0623891e-09 2.7367414e-06]


In [29]:
import os

def batch_inference(root_folder, model, class_folders, n_mfcc=13, max_len=100):
    """
    Iterate through subfolders and predict speaker for each audio file.
    """
    results = []

    # Iterate over each subfolder (ground truth class)
    for subfolder in sorted(os.listdir(root_folder)):
        subfolder_path = os.path.join(root_folder, subfolder)
        if not os.path.isdir(subfolder_path):
            continue  # skip files

        # Iterate over each audio file
        for filename in sorted(os.listdir(subfolder_path)):
            if filename.endswith((".wav", ".mp3")):
                file_path = os.path.join(subfolder_path, filename)
                
                # Predict class
                pred_class, pred_prob = predict_speaker(file_path, model, class_folders, n_mfcc, max_len)
                
                # Store results
                results.append({
                    "file": filename,
                    "ground_truth": subfolder,
                    "predicted": pred_class
                })
                
                print(f"File: {filename} | Ground Truth: {subfolder} | Predicted: {pred_class}")
    
    return results


In [30]:
root_test_folder = "/home/shanin/Downloads/test"  # root folder with subfolders
results = batch_inference(root_test_folder, model, class_folders, n_mfcc=13, max_len=100)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
File: 495.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
File: 496.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
File: 497.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
File: 498.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
File: 499.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
File: 500.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
File: 501.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
File: 502.wav | Ground Truth: Benjamin_Netanyau | Predicted: Benjamin_Netanyau
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
File: 503.wav | Ground Tru

In [31]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results


,file,ground_truth,predicted
0,495.wav,Benjamin_Netanyau,Benjamin_Netanyau
1,496.wav,Benjamin_Netanyau,Benjamin_Netanyau
2,497.wav,Benjamin_Netanyau,Benjamin_Netanyau
3,498.wav,Benjamin_Netanyau,Benjamin_Netanyau
4,499.wav,Benjamin_Netanyau,Benjamin_Netanyau
5,500.wav,Benjamin_Netanyau,Benjamin_Netanyau
6,501.wav,Benjamin_Netanyau,Benjamin_Netanyau
7,502.wav,Benjamin_Netanyau,Benjamin_Netanyau
8,503.wav,Benjamin_Netanyau,Benjamin_Netanyau
9,504.wav,Benjamin_Netanyau,Benjamin_Netanyau


# SpeechBrain (Colab)

In [ ]:
"""
Speaker Recognition using SpeechBrain ECAPA-TDNN
------------------------------------------------
• Converts MP3 → WAV (16kHz mono)
• Builds speaker database from folder
• Tests unknown audio
• Prints similarity scores
"""

import os
from speechbrain.pretrained import SpeakerRecognition
from pydub import AudioSegment

# -----------------------------
# CONFIGURATION
# -----------------------------
SPEAKER_FOLDER = "speaker_audio"       # Folder of training audio
TEST_AUDIO = "test_audio/test.mp3"     # File to identify
THRESHOLD = 0.10                       # Unknown speaker threshold
SR = 16000                             # Target sample rate
# -----------------------------


# Convert MP3 to WAV
def convert_to_wav(input_path):
    if input_path.lower().endswith(".wav"):
        return input_path

    sound = AudioSegment.from_file(input_path)
    sound = sound.set_channels(1)
    sound = sound.set_frame_rate(SR)

    wav_path = input_path.rsplit(".", 1)[0] + ".wav"
    sound.export(wav_path, format="wav")
    return wav_path


# Extract speaker name from filename
def get_speaker_name(filename):
    # Examples:
    # aiyub_01.mp3 -> aiyub
    # Hasan1.mp3 -> Hasan
    base = os.path.splitext(filename)[0]
    name = ''.join([c for c in base if not c.isdigit()]).strip("_")
    return name.capitalize()


# Build speaker database
def build_speaker_db(folder):
    speaker_db = {}
    for file in os.listdir(folder):
        if file.endswith((".mp3", ".wav")):
            path = os.path.join(folder, file)
            wav_path = convert_to_wav(path)
            speaker_name = get_speaker_name(file)
            speaker_db[speaker_name] = wav_path
            print(f"[DB] {speaker_name} ← {wav_path}")

    print(f"\n✅ Loaded {len(speaker_db)} speakers.")
    return speaker_db


# Identify speaker
def identify_speaker(model, test_file, speaker_db, threshold):
    test_wav = convert_to_wav(test_file)

    scores = {}
    for name, ref in speaker_db.items():
        score, _ = model.verify_files(test_wav, ref)
        scores[name] = score.item()

    identified = max(scores, key=scores.get)
    best_score = scores[identified]

    print("\n🔍 Similarity scores:")
    for k, v in scores.items():
        print(f"  {k}: {v:.3f}")

    print("\n🎯 Result:")
    if best_score < threshold:
        print(f"❌ Unknown speaker (best score={best_score:.3f})")
    else:
        print(f"✅ Identified: {identified} (score={best_score:.3f})")

    return identified, best_score


# ------------------- MAIN ----------------------- #

print("🚀 Loading ECAPA-TDNN model...")
model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_ecapa"
)
print("✅ Model Loaded!")

print("\n📂 Building speaker database...")
speaker_db = build_speaker_db(SPEAKER_FOLDER)

print("\n🎤 Identifying test speaker...")
identify_speaker(model, TEST_AUDIO, speaker_db, THRESHOLD)
